# Cold-Start Lab — Distributed Fleet Worker

Run this **same notebook in N Colab sessions at once**. They coordinate through one
Postgres ledger: each session claims tasks nobody else is running, benchmarks them, and
commits results. No session needs to know about the others.

**Exactly-once execution is a correctness requirement here, not an optimisation.** Two
sessions running the same task would contend for the same disk and corrupt the very I/O
measurement we're taking. The coordinator guarantees it with an atomic conditional
UPDATE plus epoch fencing — see `distributed/coordinator.py`.

## Order of operations
1. Every session: run cells 1–3 (setup, credentials, hardware).
2. **One** session only: run cell 4 (`init`) to register the task list.
3. Every session: run cell 5 (`work`). Order doesn't matter after init.
4. Anywhere: cell 6 to watch progress, cell 7 to merge results.

## 1. Setup
Upload `coldstart-lab.zip` to `/content` first (Files pane → Upload). The setup script
handles nested folders, `__MACOSX` sidecars and re-uploads.

In [ ]:
!unzip -o -q /content/coldstart-lab.zip -d /content && bash $(find /content -name colab_setup.sh | head -1)
%cd /content/coldstart-lab

## 2. Database credentials
`getpass` keeps the connection string out of the notebook's saved output — a plain
assignment cell would persist it into the `.ipynb` and into any share link.

Paste your Neon URL. The helper rewrites it for SQLAlchemy automatically (adds the
psycopg2 driver, drops `channel_binding`, keeps `sslmode=require`).

In [ ]:
import os, getpass
os.environ['COLDSTART_DB_URL'] = getpass.getpass('Postgres URL: ')

from coldstart_lab.distributed import get_db_url, redact
print('connecting as:', redact(get_db_url()))

## 3. What hardware am I?
`device_class` becomes part of the task identity: a cold-start number from a T4 is not
interchangeable with one from an A100, so the same experiment on different hardware is a
*different* task, not a duplicate.

In [ ]:
from coldstart_lab import environment
fp = environment.probe()
DEVICE = 'cuda' if fp.cuda_available else 'cpu'
gpu = fp.gpus[0].name if fp.gpus else 'cpu'
DEVICE_CLASS = gpu.lower().replace('nvidia ','').replace(' ','-').replace('tesla-','') if fp.gpus else 'cpu'
print(f'device={DEVICE}  gpu={gpu}  device_class={DEVICE_CLASS}')
if fp.warnings:
    print('\nwarnings:'); [print(' -', w) for w in fp.warnings]

## 4. Register the task list — RUN IN ONE SESSION ONLY
`register()` is idempotent, so running it twice is harmless (it inserts 0 new rows) — but
there's no reason to. Pick the tier that matches your GPU:
`micro` for CPU, `small` for a T4, `medium` for a 24 GB card.\n\nEvery model in the registry is **ungated** — no HF token or licence acceptance is needed for any of them.

In [ ]:
# 61 models across 5 tiers. Pick what your GPU can hold:
#   micro (<1 GiB, CPU) | small (<=7.5 GiB, T4) | medium (<=23 GiB, L4/A100)
# --max-gib trims a tier to fit a smaller card or disk.
TIER = 'micro' if DEVICE == 'cpu' else 'small'
!coldstart-fleet init --tier $TIER --device-class $DEVICE_CLASS

# Examples:
#   !coldstart-fleet init --tier small --device-class t4 --max-gib 6
#   !coldstart-fleet init --tier medium --device-class a100 --family qwen2.5 qwen2.5-awq


## 5. Work — RUN IN EVERY SESSION
Claims one task at a time and runs it to completion. A background thread refreshes the
lease every 60s; if this session is pre-empted the lease expires and another worker picks
the task up. Safe to re-run after a disconnect — finished tasks are never redone.

In [ ]:
# Workers wait for in-flight tasks instead of exiting on an empty queue,
# so it is safe (and correct) to start these in any order.
!coldstart-fleet work --device $DEVICE --device-class $DEVICE_CLASS --repeats 5


## 6. Watch the fleet
Run this in any session (including a spare one) to see live progress.

In [ ]:
from coldstart_lab.distributed import Coordinator, get_db_url
coord = Coordinator(get_db_url())
print('progress:', coord.progress())
print()
for row in sorted(coord.status(), key=lambda r: r['task_id']):
    print(f"  {row['task_id']:<48} {row['status']:<8} {row['owner'] or '-'}")

## 7. Merge and report
Pulls every finished result out of the ledger into one JSON, then builds the combined
Markdown report with production-scale projections.

In [ ]:
# Writes merged_results.json AND cross_model_report.md (the analysis).
!coldstart-fleet merge --out /content/merged

from IPython.display import Markdown, display
display(Markdown(open('/content/merged/cross_model_report.md').read()))


## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `COLDSTART_DB_URL is not set` | cell 2 not run in this session | run cell 2 |
| worker exits with `completed:0, failed:0` | **fixed** — workers now wait for in-flight tasks. Check `fleet_finished` in the summary | if `fleet_finished:false`, just re-run the work cell |
| `GatedRepoError 401` | a gated repo | no longer possible; the registry is 100% ungated |
| `rejected_stale > 0` | session paused past its lease | normal; another worker redid it |
| `No space left on device` | staged copies of big checkpoints | staging is now cleared after every task; also `!rm -rf ~/.cache/huggingface` |
| tasks stuck in `running` with a dead owner | lease not yet expired | wait `LEASE_TIMEOUT_S` (1800s); any worker reaps it |
| some tasks `failed` | see the reason | `!coldstart-fleet status` lists each error; `!coldstart-fleet retry` re-queues them |
